In [0]:
parquet_path = "/Volumes/workspace/default/nyc_311_data/"

df = spark.read.parquet(parquet_path)

In [0]:
print(f"Row count: {df.count()}")
df.printSchema()

Row count: 389087
root
 |-- Unique Key: integer (nullable = true)
 |-- Created Date: timestamp (nullable = true)
 |-- Closed Date: timestamp (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Agency Name: string (nullable = true)
 |-- Problem (formerly Complaint Type): string (nullable = true)
 |-- Problem Detail (formerly Descriptor): string (nullable = true)
 |-- Additional Details: string (nullable = true)
 |-- Location Type: string (nullable = true)
 |-- Incident Zip: string (nullable = true)
 |-- Incident Address: string (nullable = true)
 |-- Street Name: string (nullable = true)
 |-- Cross Street 1: string (nullable = true)
 |-- Cross Street 2: string (nullable = true)
 |-- Intersection Street 1: string (nullable = true)
 |-- Intersection Street 2: string (nullable = true)
 |-- Address Type: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Landmark: string (nullable = true)
 |-- Facility Type: string (nullable = true)
 |-- Status: string (nullable = tr

In [0]:
df.select(
    "request_date",
    "request_day_of_week",
    "request_hour",
    "resolution_time_hours"
).show(5, truncate=False)

+------------+-------------------+------------+---------------------+
|request_date|request_day_of_week|request_hour|resolution_time_hours|
+------------+-------------------+------------+---------------------+
|2025-12-22  |Monday             |11          |31.77                |
|2025-12-22  |Monday             |11          |3.03                 |
|2025-12-22  |Monday             |11          |2750.3               |
|2025-12-22  |Monday             |11          |0.52                 |
|2025-12-22  |Monday             |11          |1.35                 |
+------------+-------------------+------------+---------------------+
only showing top 5 rows


In [0]:
df.createOrReplaceTempView("nyc_311")

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM nyc_311;

total_rows
389087


In [0]:
%sql
-- Top Complaint Types Within Each Borough and Their Percentage Share
WITH complaint_count AS (
    SELECT
        N.Borough,
        N.`Problem (formerly Complaint Type)`,
        COUNT(*) AS num_complaints
    FROM nyc_311 N
    WHERE N.Borough <> 'Unspecified'
    GROUP BY N.Borough, N.`Problem (formerly Complaint Type)`
),

ranked_complaints AS (
    SELECT
        CC.Borough,
        CC.`Problem (formerly Complaint Type)`,
        CC.num_complaints,
        SUM(CC.num_complaints) OVER(
            PARTITION BY CC.Borough
        ) AS borough_total,
        RANK() OVER (
            PARTITION BY CC.Borough ORDER BY CC.num_complaints DESC
        ) AS complaint_rank
    FROM complaint_count CC
)

SELECT
    RC.Borough,
    RC.`Problem (formerly Complaint Type)`,
    RC.num_complaints,
    RC.complaint_rank,
    ROUND(
        (RC.num_complaints * 100.0) / RC.borough_total, 2
    ) AS complaint_share_percentage
FROM ranked_complaints RC
WHERE RC.complaint_rank <= 3
ORDER BY RC.Borough, RC.complaint_rank;

Borough,Problem (formerly Complaint Type),num_complaints,complaint_rank,complaint_share_percentage
BRONX,Noise - Residential,47082,1,40.76
BRONX,HEAT/HOT WATER,27016,2,23.39
BRONX,Illegal Parking,7682,3,6.65
BROOKLYN,Illegal Parking,20329,1,18.85
BROOKLYN,HEAT/HOT WATER,20068,2,18.61
BROOKLYN,Noise - Residential,10123,3,9.39
MANHATTAN,HEAT/HOT WATER,18645,1,25.95
MANHATTAN,Noise - Residential,6829,2,9.51
MANHATTAN,Illegal Parking,5631,3,7.84
QUEENS,Illegal Parking,16371,1,20.17


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- High-Volume and Slow-Resolution Complaint Types
WITH request_count AS (
    SELECT
        N.`Problem (formerly Complaint Type)`,
        COUNT(*) AS total_request_count,
        COUNT(
            CASE 
                WHEN N.resolution_time_hours IS NOT NULL THEN 1 END
        ) AS resolved_request_count,
        ROUND(
            AVG(N.resolution_time_hours), 2
        ) AS average_resolution_time_hours
    FROM nyc_311 N
    GROUP BY N.`Problem (formerly Complaint Type)`
)

SELECT
    RC.`Problem (formerly Complaint Type)`,
    RC.total_request_count,
    RC.resolved_request_count,
    RC.average_resolution_time_hours
FROM request_count RC
WHERE RC.total_request_count > (
    SELECT AVG(total_request_count)
    FROM request_count
)
ORDER BY RC.average_resolution_time_hours DESC;

Problem (formerly Complaint Type),total_request_count,resolved_request_count,average_resolution_time_hours
GENERAL,3656,3577,1052.14
UNSANITARY CONDITION,11084,10902,954.19
WATER LEAK,4307,4262,902.81
DOOR/WINDOW,5350,5236,888.29
FLOORING/STAIRS,2587,2551,824.6
Plumbing,8142,7991,811.39
Elevator,2312,2307,771.48
ELECTRIC,3275,3218,769.74
General Construction/Plumbing,2884,2823,688.73
PAINT/PLASTER,5846,5780,681.04


In [0]:
%sql
-- Agency Workload vs Resolution Speed
WITH agency_count AS (
    SELECT
        N.Agency,
        COUNT(*) AS total_requests,
        COUNT(
            CASE
                WHEN N.resolution_time_hours IS NOT NULL THEN 1 END
        ) AS resolved_requests,
        ROUND(
            AVG(N.resolution_time_hours), 2
        ) AS avg_resolution_hours
    FROM nyc_311 N
    GROUP BY N.Agency
),

ranked_agencies AS (
    SELECT
        AC.Agency,
        AC.total_requests,
        AC.resolved_requests,
        AC.avg_resolution_hours,
        RANK() OVER (
            ORDER BY AC.total_requests DESC
        ) AS workload_rank,
        CASE
            WHEN AC.avg_resolution_hours IS NOT NULL THEN
                RANK() OVER (
                    ORDER BY AC.avg_resolution_hours ASC NULLS LAST
                )
        END AS speed_rank
    FROM agency_count AC
)

SELECT
    RA.Agency,
    RA.total_requests,
    RA.resolved_requests,
    RA.avg_resolution_hours,
    RA.workload_rank,
    RA.speed_rank
FROM ranked_agencies RA
ORDER BY RA.workload_rank;

Agency,total_requests,resolved_requests,avg_resolution_hours,workload_rank,speed_rank
NYPD,171981,171931,2.7,1,1
HPD,123899,123039,360.41,2,6
DSNY,29786,29572,82.93,3,2
DOT,18346,17618,209.16,4,5
DEP,14340,14115,101.09,5,4
DOB,8950,8717,990.32,6,13
DOHMH,5906,5711,662.98,7,10
DPR,5718,4840,537.59,8,8
TLC,3512,2597,2189.37,9,14
DHS,2701,2700,90.89,10,3


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Borough Resolution Coverage and Speed
WITH borough_resolution AS (
    SELECT
        N.Borough,
        COUNT(*) AS total_request_count,
        COUNT(
            CASE
                WHEN N.resolution_time_hours IS NOT NULL THEN 1 END
        ) AS resolved_request_count,
        ROUND(
            AVG(N.resolution_time_hours), 2
        ) AS avg_resolution_time_hours
    FROM nyc_311 N
    WHERE N.Borough <> 'Unspecified'
    GROUP BY N.Borough
)

SELECT
    BR.Borough,
    BR.total_request_count,
    BR.resolved_request_count,
    BR.avg_resolution_time_hours,
    ROUND(
        (BR.resolved_request_count * 100.0) / BR.total_request_count, 2
    ) AS resolution_coverage_percentage
FROM borough_resolution BR
ORDER BY BR.avg_resolution_time_hours DESC;

Borough,total_request_count,resolved_request_count,avg_resolution_time_hours,resolution_coverage_percentage
MANHATTAN,71836,69576,285.22,96.85
BROOKLYN,107846,106163,193.29,98.44
QUEENS,81148,79724,179.44,98.25
STATEN ISLAND,12492,12268,165.29,98.21
BRONX,115506,114941,152.77,99.51


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Peak Hours for 311 Activity
WITH hourly_count AS (
    SELECT 
        N.request_hour,
        COUNT(*) AS hourly_request_count
    FROM nyc_311 N
    GROUP BY N.request_hour
),

hourly_ranks AS (
    SELECT
        HC.request_hour,
        HC.hourly_request_count,
        SUM(HC.hourly_request_count) OVER() AS total_requests,
        DENSE_RANK() OVER(
            ORDER BY HC.hourly_request_count DESC
        ) AS request_hour_rank
    FROM hourly_count HC
)

SELECT
    CONCAT(
        LPAD(
            CAST(HR.request_hour AS STRING), 2, '0'), ':00'
    ) AS request_hour,
    HR.hourly_request_count,
    HR.request_hour_rank,
    ROUND(
        (HR.hourly_request_count * 100.0) / HR.total_requests, 2
    ) AS hour_share_percentage
FROM hourly_ranks HR
ORDER BY HR.request_hour;

request_hour,hourly_request_count,request_hour_rank,hour_share_percentage
00:00,13454,18,3.46
01:00,9795,19,2.52
02:00,7529,21,1.94
03:00,6515,23,1.67
04:00,6216,24,1.60
05:00,6846,22,1.76
06:00,9235,20,2.37
07:00,15411,16,3.96
08:00,20678,7,5.31
09:00,23233,2,5.97


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Day of Week and Weekday/Weekend Pattern
WITH date_bounds AS (
    SELECT
        MIN(N.request_date) AS first_date,
        MAX(N.request_date) AS last_date
    FROM nyc_311 N
),

daily_requests AS (
    SELECT
        N.request_date,
        N.request_day_of_week,
        COUNT(*) AS daily_request_count,
        ROUND(
            AVG(N.resolution_time_hours), 2
        ) AS average_resolution_hours
    FROM nyc_311 N
    CROSS JOIN date_bounds DB
    WHERE N.request_date > DB.first_date
        AND N.request_date < DB.last_date
    GROUP BY N.request_date, N.request_day_of_week
),

request_volume AS (
    SELECT
        DR.request_day_of_week,
        CASE
            WHEN DR.request_day_of_week IN ('Saturday', 'Sunday') THEN 'Weekend'
            ELSE 'Weekday'
        END AS day_type,
        ROUND(
            AVG(DR.daily_request_count), 2
        ) AS average_requests_per_day,
        ROUND(
            AVG(DR.average_resolution_hours), 2
        ) AS average_resolution_hours,
        RANK() OVER (
            ORDER BY AVG(DR.daily_request_count) DESC
        ) AS day_rank
    FROM daily_requests DR
    GROUP BY DR.request_day_of_week
)

SELECT
    RV.request_day_of_week,
    RV.day_type,
    RV.average_requests_per_day,
    RV.average_resolution_hours,
    RV.day_rank
FROM request_volume RV
ORDER BY
    CASE
        WHEN RV.request_day_of_week = 'Sunday' THEN 1
        WHEN RV.request_day_of_week = 'Monday' THEN 2
        WHEN RV.request_day_of_week = 'Tuesday' THEN 3
        WHEN RV.request_day_of_week = 'Wednesday' THEN 4
        WHEN RV.request_day_of_week = 'Thursday' THEN 5
        WHEN RV.request_day_of_week = 'Friday' THEN 6
        WHEN RV.request_day_of_week = 'Saturday' THEN 7
    END;

request_day_of_week,day_type,average_requests_per_day,average_resolution_hours,day_rank
Sunday,Weekend,9697.2,146.84,6
Monday,Weekday,12139.2,230.26,1
Tuesday,Weekday,11735.2,224.16,2
Wednesday,Weekday,10345.6,199.51,4
Thursday,Weekday,9771.0,179.0,5
Friday,Weekday,10825.2,208.95,3
Saturday,Weekend,9614.8,162.8,7


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Daily Spike vs Daily Average
WITH request_count AS (
    SELECT
        N.request_date,
        COUNT(*) AS daily_request_count
    FROM nyc_311 N
    GROUP BY N.request_date
),

avg_requests AS (
    SELECT
        ROUND(
            AVG(RC.daily_request_count), 2
        ) AS avg_daily_requests
    FROM request_count RC
)

SELECT
    RC.request_date,
    RC.daily_request_count,
    AR.avg_daily_requests,
    ROUND(
        (RC.daily_request_count - AR.avg_daily_requests) / NULLIF(AR.avg_daily_requests, 0) * 100.0, 2
    ) AS percentage_difference,
    CASE
        WHEN RC.daily_request_count > AR.avg_daily_requests THEN 'Above Average'
        WHEN RC.daily_request_count < AR.avg_daily_requests THEN 'Below Average'
        ELSE 'Average'
    END AS comparison_status
FROM request_count RC
CROSS JOIN avg_requests AR
ORDER BY RC.request_date;

request_date,daily_request_count,avg_daily_requests,percentage_difference,comparison_status
2025-11-27,8147,10515.86,-22.53,Below Average
2025-11-28,10282,10515.86,-2.22,Below Average
2025-11-29,10093,10515.86,-4.02,Below Average
2025-11-30,9872,10515.86,-6.12,Below Average
2025-12-01,12302,10515.86,16.99,Above Average
2025-12-02,12226,10515.86,16.26,Above Average
2025-12-03,12507,10515.86,18.93,Above Average
2025-12-04,13118,10515.86,24.74,Above Average
2025-12-05,13129,10515.86,24.85,Above Average
2025-12-06,11185,10515.86,6.36,Above Average


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Peak Hour By Complaint Type
WITH top_complaints AS (
    SELECT
        N.`Problem (formerly Complaint Type)`,
        COUNT(*) AS total_requests,
        RANK() OVER (
            ORDER BY COUNT(*) DESC
        ) AS complaint_rank
    FROM nyc_311 N
    GROUP BY N.`Problem (formerly Complaint Type)`
),

complaint_hours AS (
    SELECT
        N.`Problem (formerly Complaint Type)`,
        N.request_hour,
        COUNT(*) AS hourly_request_count,
        DENSE_RANK() OVER (
            PARTITION BY N.`Problem (formerly Complaint Type)`
            ORDER BY COUNT(*) DESC
        ) AS hour_rank
    FROM nyc_311 N
    JOIN top_complaints TC
        ON N.`Problem (formerly Complaint Type)` = TC.`Problem (formerly Complaint Type)`
    WHERE TC.complaint_rank <= 10
    GROUP BY N.`Problem (formerly Complaint Type)`, N.request_hour
)

SELECT
    CH.`Problem (formerly Complaint Type)`,
    CONCAT(
        LPAD(
            CAST(CH.request_hour AS STRING), 2, '0'), ':00'
    ) AS request_hour,
    CH.hourly_request_count,
    CH.hour_rank
FROM complaint_hours CH
WHERE CH.hour_rank <= 3
ORDER BY CH.`Problem (formerly Complaint Type)`, CH.hour_rank;

Problem (formerly Complaint Type),request_hour,hourly_request_count,hour_rank
Blocked Driveway,20:00,1162,1
Blocked Driveway,21:00,1159,2
Blocked Driveway,19:00,1122,3
HEAT/HOT WATER,16:00,4556,1
HEAT/HOT WATER,10:00,4520,2
HEAT/HOT WATER,09:00,4413,3
Illegal Parking,21:00,3424,1
Illegal Parking,08:00,2988,2
Illegal Parking,09:00,2949,3
Noise - Residential,23:00,4929,1


In [0]:
%sql
-- Complaint Resolution Differences Across Boroughs
WITH top_complaints AS (
    SELECT
        N.`Problem (formerly Complaint Type)`,
        COUNT(*) AS total_requests,
        RANK() OVER (
            ORDER BY COUNT(*) DESC
        ) AS complaint_rank
    FROM nyc_311 N
    GROUP BY N.`Problem (formerly Complaint Type)`
),

complaint_borough AS (
    SELECT
        N.Borough,
        N.`Problem (formerly Complaint Type)`,
        COUNT(
            CASE
                WHEN N.resolution_time_hours IS NOT NULL THEN 1 END
        ) AS resolved_requests,
        ROUND(
            AVG(N.resolution_time_hours), 2
        ) AS avg_resolution_hours,
        DENSE_RANK() OVER (
            PARTITION BY N.`Problem (formerly Complaint Type)`
            ORDER BY AVG(N.resolution_time_hours) DESC
        ) AS resolution_rank
    FROM nyc_311 N
    JOIN top_complaints TC
        ON N.`Problem (formerly Complaint Type)` = TC.`Problem (formerly Complaint Type)`
    WHERE TC.complaint_rank <= 10
        AND N.Borough <> 'Unspecified'
    GROUP BY
        N.Borough,
        N.`Problem (formerly Complaint Type)`
    HAVING COUNT(
        CASE
            WHEN N.resolution_time_hours IS NOT NULL THEN 1 END
    ) >= 100
)

SELECT
    CB.`Problem (formerly Complaint Type)`,
    CB.Borough,
    CB.resolved_requests,
    CB.avg_resolution_hours,
    CB.resolution_rank
FROM complaint_borough CB
ORDER BY
    CB.`Problem (formerly Complaint Type)`,
    CB.resolution_rank;

Problem (formerly Complaint Type),Borough,resolved_requests,avg_resolution_hours,resolution_rank
Blocked Driveway,QUEENS,8175,3.03,1
Blocked Driveway,BROOKLYN,7196,2.9,2
Blocked Driveway,BRONX,2929,2.57,3
Blocked Driveway,STATEN ISLAND,594,2.54,4
Blocked Driveway,MANHATTAN,482,2.0,5
HEAT/HOT WATER,QUEENS,9814,53.03,1
HEAT/HOT WATER,BROOKLYN,20060,49.79,2
HEAT/HOT WATER,MANHATTAN,18608,47.33,3
HEAT/HOT WATER,STATEN ISLAND,606,43.42,4
HEAT/HOT WATER,BRONX,27012,41.23,5
